In [1]:
# ================= Parte a) Método de Froude / ITTC-1957 =================
# Linha de correlacao ITTC-1957 para o coeficiente de atrito (placa plana equivalente)
import numpy as np

g = 9.81

def Cf(Rex):
    return 0.075 / (np.log10(Rex) - 2) ** 2

In [2]:
# Dados do navio em escala real (dados do enunciado)
ship = {
    "DWT": 9e3,        # porte bruto [t]
    "C_DWT": 0.69,     # relacao DWT/deslocamento
    "Vs": 14*0.5144,   # velocidade de servico [m/s] (14,0 nos)
    "rho": 1.025,      # massa especifica da agua do mar a 19 C [t/m3]
    "mu": 1.07854e-6,  # viscosidade cinematica da agua do mar a 19 C [m2/s]
    "rug": 1e6
}

# Deslocamento e volume deslocado do navio
ship["Delta"] = ship["DWT"] / ship["C_DWT"]
ship["Vd"] = ship["Delta"] / ship["rho"]

print(ship["Delta"], ship["Vd"])

13043.478260869566 12725.344644750798


In [3]:
# Dados do modelo ensaiado no tanque de provas (dados do enunciado)
model = {
    "LBP": 4.6,        # comprimento entre perpendiculares [m]
    "B": 4.6 / 6.3,    # boca [m], a partir de L/B = 6,3
    "Cb": 0.82,        # coeficiente de bloco
    "S": 5.5041,       # area molhada [m2]
    "Vd": 0.8758,      # volume deslocado (Nabla) [m3]
    "Ct": 0.00611,     # Ct ensaiado no Fn de servico
    "rho": 0.9987,     # massa especifica da agua doce a 17 C [t/m3]
    "mu": 1.08155e-6   # viscosidade cinematica da agua doce a 17 C [m2/s]
}

In [4]:
# Fator de escala geometrica, a partir da razao de volumes deslocados (Vd ~ lambda^3)
F_scale = (ship["Vd"] / model["Vd"])**(1/3)
print(F_scale)

24.40178169478108


In [5]:
# Comprimento e numero de Reynolds do navio em escala real, na velocidade de servico
ship["LBP"] = model["LBP"]*F_scale
ship["Re"] = ship["rho"]*ship["Vs"]*ship["LBP"]/ship["mu"]

print(ship["LBP"], ship["Re"])

112.24819579599296 768238333.3168296


In [6]:
# Velocidade e Reynolds do modelo no mesmo Fn do navio (similaridade de Froude)
# Fn_navio = Fn_modelo  =>  Vs_modelo = Vs_navio * sqrt(L_modelo/L_navio)
model["Vs"] = ship["Vs"] * np.sqrt(model["LBP"] / ship["LBP"])
model["Re"] = model["rho"]*model["Vs"]*model["LBP"]/model["mu"]

print(model["Vs"], model["Re"])

1.4578680788596072 6192478.490724065


In [7]:
# Coeficiente de atrito (ITTC-1957), aplicado separadamente ao navio e ao modelo via seus respectivos Re
ship["Cf"] = Cf(ship["Re"])
model["Cf"] = Cf(model["Re"])

print(ship["Cf"], model["Cf"])

0.0015819429087601076 0.0032662709191678143


In [8]:
# Coeficiente de resistência residual (Cr) do modelo, obtido do ensaio
# Cr nao depende da escala (hipotese de Froude -> mesmo Fn, mesmo Cr)
model["Cr"] = model["Ct"] - model["Cf"]
ship["Cr"] = model["Cr"]
print(model["Cr"], ship["Cr"])

0.0028437290808321857 0.0028437290808321857


In [9]:
# Pontos conhecidos (Ca em funcao do deslocamento do navio)
x = [1e3, 1e4, 1e5, 1e6]
y = [6e-4, 4e-4, 0.0, -6e-4]

# Ponto(s) onde você deseja descobrir o valor de y
x_interp = ship["Delta"]

# Calculando a interpolação linear
ship["Ca"] = np.interp(x_interp, x, y)
model["Ca"] = 0.0  # nao se aplica correlacao modelo-navio ao proprio modelo
print(ship["Ca"])

0.0003864734299516908


In [10]:
# Extrapolação ITTC-1957: Ct_navio = Cr + Cf_navio + Ca
ship["Ct"] = ship["Cr"] + ship["Cf"] + ship["Ca"]
print(ship["Ct"])

0.004812145419543984


In [11]:
# Resumo dos coeficientes e grandezas calculados para o navio (metodo de Froude / ITTC-1957)
ship

{'DWT': 9000.0,
 'C_DWT': 0.69,
 'Vs': 7.201599999999999,
 'rho': 1.025,
 'mu': 1.07854e-06,
 'rug': 1000000.0,
 'Delta': 13043.478260869566,
 'Vd': 12725.344644750798,
 'LBP': 112.24819579599296,
 'Re': 768238333.3168296,
 'Cf': np.float64(0.0015819429087601076),
 'Cr': np.float64(0.0028437290808321857),
 'Ca': np.float64(0.0003864734299516908),
 'Ct': np.float64(0.004812145419543984)}

In [12]:
# Resumo dos coeficientes e grandezas calculados para o modelo (metodo de Froude / ITTC-1957)
model

{'LBP': 4.6,
 'B': 0.7301587301587301,
 'Cb': 0.82,
 'S': 5.5041,
 'Vd': 0.8758,
 'Ct': 0.00611,
 'rho': 0.9987,
 'mu': 1.08155e-06,
 'Vs': np.float64(1.4578680788596072),
 'Re': np.float64(6192478.490724065),
 'Cf': np.float64(0.0032662709191678143),
 'Cr': np.float64(0.0028437290808321857),
 'Ca': 0.0}

In [13]:
# Area molhada do navio (escala geometrica F_scale^2)
ship["S"] = model["S"] * F_scale**2
print(ship["S"])

3277.399556833148


In [14]:
# Resistencia total ao avanco do navio em escala real
# rho em t/m3 -> *1000 para kg/m3, Rt em N
ship["Rt"] = ship["Ct"] * 0.5 * (ship["rho"]*1000) * ship["Vs"]**2 * ship["S"]
print(f"Rt_navio = {ship['Rt']/1000:.2f} kN")

Rt_navio = 419.20 kN


In [15]:
# ================= Parte b) Método ITTC-1978 =================
# Ct = (1+k)*Cf + Cw  (decomposicao conceitual, separa efeito de forma do casco do atrito de placa plana)
# k = fator de forma, obtido de ensaio extra em baixo Fn (Fn=0.1), onde Cw ~ 0

ittc1978 = {"h": 150e-6}  # rugosidade padrao [m]

# Condicao do ensaio extra do modelo a Fn = 0.1
ittc1978["Fn_k"] = 0.1
ittc1978["V_model_k"] = ittc1978["Fn_k"] * np.sqrt(g * model["LBP"])
ittc1978["Re_model_k"] = model["rho"] * ittc1978["V_model_k"] * model["LBP"] / model["mu"]
ittc1978["Cf_model_k"] = Cf(ittc1978["Re_model_k"])
ittc1978["Ct_model_k"] = 0.004487  # ensaiado a Fn=0.1

print(ittc1978["V_model_k"], ittc1978["Re_model_k"], ittc1978["Cf_model_k"])

0.6717588853152595 2853380.5689205388 0.0037782937429882302


In [16]:
# Fator de forma k: a Fn=0.1 a geracao de ondas e desprezivel (Cw ~ 0)
# Ct_model_k = (1+k)*Cf_model_k  =>  k = Ct_model_k/Cf_model_k - 1
ittc1978["k"] = ittc1978["Ct_model_k"] / ittc1978["Cf_model_k"] - 1
print(ittc1978["k"])

0.18757309654046583


In [17]:
# Coeficiente de onda Cw, extraido do ensaio principal do modelo (mesmo Fn de servico, usado na parte a)
# Ct_model_main = (1+k)*Cf_model_main + Cw  =>  Cw = Ct_model_main - (1+k)*Cf_model_main
# Cw e assumido igual para modelo e navio (hipotese de Froude, mesmo Fn)
ittc1978["Cw"] = model["Ct"] - (1 + ittc1978["k"]) * model["Cf"]
print(ittc1978["Cw"])

0.002231064530383805


In [18]:
# Coeficiente de correlacao Ca pela formula de Townsin (ITTC-1978), calculado para o navio
# 10^3*Ca = 44*[(h/L)^(1/3) - 10*Re^(-1/3)] + 0.125
ittc1978["Ca_ship"] = (44 * ((ittc1978["h"] / ship["LBP"]) ** (1/3) - 10 * ship["Re"] ** (-1/3)) + 0.125) * 1e-3
print(ittc1978["Ca_ship"])

0.00012922495546125864


In [19]:
# Ct do navio pelo metodo ITTC-1978: Ct_ship = (1+k)*Cf_ship + Cw + Ca
ittc1978["Ct_ship"] = (1 + ittc1978["k"]) * ship["Cf"] + ittc1978["Cw"] + ittc1978["Ca_ship"]
print(ittc1978["Ct_ship"])

0.0042389623245515365


In [20]:
# Resistencia total ao avanco do navio pelo metodo ITTC-1978
ittc1978["Rt"] = ittc1978["Ct_ship"] * 0.5 * (ship["rho"]*1000) * ship["Vs"]**2 * ship["S"]
print(f"Rt_navio (ITTC-1978) = {ittc1978['Rt']/1000:.2f} kN")

Rt_navio (ITTC-1978) = 369.27 kN


In [21]:
# Comparacao entre os metodos de extrapolacao
print(f"Ct_navio Froude/ITTC-1957: {ship['Ct']:.6f}  ->  Rt = {ship['Rt']/1000:.2f} kN")
print(f"Ct_navio ITTC-1978:        {ittc1978['Ct_ship']:.6f}  ->  Rt = {ittc1978['Rt']/1000:.2f} kN")
print(f"Diferenca relativa: {(ittc1978['Rt']-ship['Rt'])/ship['Rt']*100:.2f} %")

Ct_navio Froude/ITTC-1957: 0.004812  ->  Rt = 419.20 kN
Ct_navio ITTC-1978:        0.004239  ->  Rt = 369.27 kN
Diferenca relativa: -11.91 %
